[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/simple_regression.ipynb)

# Simple linear regression full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (fitting a line, r², the t and F tests, the two intervals, residuals, leverage), **tier 3**
stretch problems (combine ideas, spot the trap, work backwards) and **tier 4**, the case: all 122 staff of Brannock
Engineering (a fictional company; our own data), and the question of whether any engineer is paid well below what
their experience predicts. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same regressions with `scipy.stats`,
`numpy` and `pandas`, so you can run them on your own data. And a few cells you can move yourself: the week you're
budgeting for, with its two intervals; and one extra week whose cost you drag, to see which points can tilt the line.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Decimals as
decimals: `0.75`, not `75%`; a p-value to four decimals is close enough. Numbers without thousands separators:
`20736`, not `20,736`. You can type the arithmetic too: `80-3.5*12`. For several answers, separate them with commas,
in the order the question lists them (the box's label shows the order). Signs and ids as they are: `++--++`,
`E001, E002`. You can also put your answer between the quotes in a cell's last line, `practice("P1.1", "")`, and run
the cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[simple regression cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/simple_regression_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the data CHALK · the fitted line SKY · residuals VIOLET · the confidence band SKY · the prediction band VIOLET ·
# the one thing to look at GOLD · a point that misleads CORAL

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(kw|minutes?|mins?|years?|thousands?|staff)\b|[$%]"
SAME = {"++−−−−++": {"++----++"}}                    # the residuals' signs, typed with a hyphen or a minus sign

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    if isinstance(want, str):
        g = got if isinstance(got, str) else f"{got:g}"
        g = re.sub(r"\s*,\s*", ", ", g).lower()
        return g == want.lower() or g in SAME.get(want.lower(), ())
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'The model and least squares',
     "q": 'A least-squares line has slope b1 = 3.5. The data have x̄ = 12 and ȳ = 80. What is the intercept b0?',
     "parts": ['b₀'],
     "answer": [38.0], "tol": [0.005], "function": '—',
     "working": 'The line passes through (x̄, ȳ): b0 = ȳ − b1 x̄ = 80 − 42 = 38'},
    {"id": 'P1.2', "tier": 1, "group": 'The model and least squares',
     "q": 'The estimated regression equation is ŷ = 25 + 1.8x. Predict y when x = 40.',
     "parts": ['ŷ'],
     "answer": [97.0], "tol": [0.005], "function": '—',
     "working": 'ŷ = 25 + 1.8 × 40 = 25 + 72 = 97'},
    {"id": 'P1.3', "tier": 1, "group": 'The model and least squares',
     "q": 'Five points: x = 1, 2, 3, 4, 5 and y = 3, 5, 4, 8, 10. Find the least-squares slope and intercept.',
     "parts": ['slope b₁', 'intercept b₀'],
     "answer": [1.7, 0.9000000000000004], "tol": [0.005, 0.005], "function": 'SLOPE, INTERCEPT',
     "working": 'x̄ = 3, ȳ = 6; Σ(x − x̄)(y − ȳ) = 6 + 1 + 0 + 2 + 8 = 17; Σ(x − x̄)² = 10; b1 = 1.7; b0 = 6 − 1.7 × 3 = 0.9'},
    {"id": 'P1.4', "tier": 1, "group": 'The model and least squares',
     "q": 'For ŷ = 20 + 3x, a point has x = 30 and y = 112. What is its residual?',
     "parts": ['residual e'],
     "answer": [2], "tol": [0.005], "function": '—',
     "working": 'ŷ = 20 + 90 = 110; residual e = y − ŷ = 112 − 110 = 2'},
    {"id": 'P1.5', "tier": 1, "group": 'The coefficient of determination',
     "q": 'A regression has SST = 500 and SSE = 125. What are SSR and r²?',
     "parts": ['SSR', 'r² (a decimal)'],
     "answer": [375, 0.75], "tol": [0.005, 0.0005], "function": '—',
     "working": 'SSR = SST − SSE = 375; r² = SSR/SST = 375/500 = 0.75'},
    {"id": 'P1.6', "tier": 1, "group": 'The coefficient of determination',
     "q": 'A regression has r² = 0.81 and a negative slope. What is the sample correlation r?',
     "parts": ['r'],
     "answer": [-0.9], "tol": [0.0005], "function": 'SQRT, CORREL',
     "working": 'r takes the sign of b1: r = −√0.81 = −0.9'},
    {"id": 'P1.7', "tier": 1, "group": 'Assumptions and testing for significance',
     "q": 'A regression on n = 10 points has SSE = 72. Find MSE and s, the standard error of the estimate.',
     "parts": ['MSE', 's'],
     "answer": [9.0, 3.0], "tol": [0.005, 0.005], "function": 'SQRT, STEYX',
     "working": 'MSE = SSE/(n − 2) = 72/8 = 9; s = √9 = 3'},
    {"id": 'P1.8', "tier": 1, "group": 'Assumptions and testing for significance',
     "q": 'A slope b1 = 2.4 has standard error s_b1 = 0.8 (n = 14). Compute t for H0: β1 = 0, and its degrees of freedom.',
     "parts": ['t', 'degrees of freedom'],
     "answer": [2.9999999999999996, 12], "tol": [0.005, 0], "function": '—',
     "working": 't = b1/s_b1 = 2.4/0.8 = 3; df = n − 2 = 12'},
    {"id": 'P1.9', "tier": 1, "group": 'Estimation and prediction',
     "q": 'n = 25, s = 4, and we estimate at x* = x̄. Find s_ŷ (for the mean of y) and s_pred (for one new y).',
     "parts": ['s_ŷ (for the mean of y)', 's_pred (for one new y)'],
     "answer": [0.8, 4.079215610874228], "tol": [0.0005, 0.0005], "function": 'SQRT',
     "working": 'At x̄ the (x* − x̄)² term is 0: s_ŷ = 4√(1/25) = 0.8; s_pred = 4√(1 + 1/25) ≈ 4.079'},
    {"id": 'P1.10', "tier": 1, "group": 'Residual analysis',
     "q": "A point has residual e = 6. The regression has s = 4, and the point's leverage is h = 0.1. Compute its standardized residual.",
     "parts": ['standardized residual'],
     "answer": [1.5811388300841898], "tol": [0.0005], "function": 'SQRT',
     "working": 'standardized residual = e/(s√(1 − h)) = 6/(4 × √0.9) = 6/3.795 ≈ 1.581: inside ±2'},
    {"id": 'P2.1', "tier": 2, "group": 'The model and least squares',
     "q": 'A print shop timed six runs. Pages (thousands): 2, 4, 5, 7, 8, 10. Minutes: 26, 37, 43, 51, 61, 68. Find the least-squares line, and say what the slope means.',
     "parts": ['slope b₁ (minutes per thousand pages)', 'intercept b₀ (minutes)'],
     "answer": [5.333333333333334, 15.666666666666664], "tol": [0.0005, 0.005], "function": 'SLOPE, INTERCEPT',
     "working": 'x̄ = 6, ȳ = 47.67; Σ(x − x̄)(y − ȳ) = 224; Σ(x − x̄)² = 42; b1 = 5.3333: about 5.33 more minutes per thousand pages; b0 = 15.667 (the set-up time, if the line holds near 0)'},
    {"id": 'P2.2', "tier": 2, "group": 'The coefficient of determination',
     "q": "For P2.1's print runs, find SST, SSE, SSR, r² and r.",
     "parts": ['SST', 'SSE', 'SSR', 'r²', 'r'],
     "answer": [1207.3333333333333, 12.66666666666665, 1194.6666666666665, 0.9895085588072887, 0.9947404479598129], "tol": [0.05, 0.05, 0.05, 0.0005, 0.0005], "function": 'DEVSQ, RSQ, CORREL',
     "working": 'SST = Σ(y − ȳ)² = 1207.33; SSE = Σ(y − ŷ)² = 12.67; SSR = 1194.67; r² = 0.9895; r = +√r² = 0.9947'},
    {"id": 'P2.3', "tier": 2, "group": 'Assumptions and testing for significance',
     "q": "For P2.1's print runs, test H0: β1 = 0 at α = 0.05: give s, s_b1, t and the p-value.",
     "parts": ['s', 's_b₁', 't', 'p-value (two-tailed)'],
     "answer": [1.7795130420052174, 0.2745848236826378, 19.423263317340304, 4.1421584296541525e-05], "tol": [0.0005, 0.0005, 0.005, 5e-05], "function": 'STEYX, DEVSQ, T.DIST.2T',
     "working": 's = √(SSE/4) = 1.7795; s_b1 = s/√42 = 0.2746; t = 19.42, df 4; p = T.DIST.2T(t, 4) ≈ 0.00004: reject H0'},
    {"id": 'P2.4', "tier": 2, "group": 'Assumptions and testing for significance',
     "q": 'A regression on n = 18 points gives b1 = 0.85 with s_b1 = 0.21. Give the 95% confidence interval for β1.',
     "parts": ['95% interval: lower end', '95% interval: upper end'],
     "answer": [0.4048198871635366, 1.2951801128364633], "tol": [0.0005, 0.0005], "function": 'T.INV.2T',
     "working": 't = T.INV.2T(0.05, 16) ≈ 2.120; 0.85 ± 2.120 × 0.21 = 0.85 ± 0.445: about 0.405 to 1.295; zero is outside, so the slope is significant at 0.05'},
    {"id": 'P2.5', "tier": 2, "group": 'Assumptions and testing for significance',
     "q": 'A regression on n = 20 points has SSR = 840 and SSE = 360. Build the ANOVA table: MSR, MSE, F and the p-value.',
     "parts": ['MSR', 'MSE', 'F', 'p-value'],
     "answer": [840.0, 20.0, 42.0, 4.274848617979998e-06], "tol": [0.005, 0.005, 0.005, 1e-06], "function": 'F.DIST.RT',
     "working": 'MSR = 840/1 = 840; MSE = 360/18 = 20; F = 840/20 = 42; p = F.DIST.RT(42, 1, 18) ≈ 0.0000043: reject H0'},
    {"id": 'P2.6', "tier": 2, "group": 'Estimation and prediction',
     "q": 'n = 15, ŷ = 5 + 2x, x̄ = 10, Σ(x − x̄)² = 160, s = 3. Give the 95% confidence interval for the mean of y at x* = 14.',
     "parts": ['95% interval: lower end', '95% interval: upper end'],
     "answer": [30.354099567682056, 35.645900432317944], "tol": [0.005, 0.005], "function": 'T.INV.2T, SQRT',
     "working": 'ŷ = 33; s_ŷ = 3√(1/15 + 16/160) = 3√0.1667 ≈ 1.2247; t = 2.160; 33 ± 2.646'},
    {"id": 'P2.7', "tier": 2, "group": 'Estimation and prediction',
     "q": 'Same regression as P2.6: give the 95% prediction interval for one new y at x* = 14, and say why it is wider.',
     "parts": ['95% interval: lower end', '95% interval: upper end'],
     "answer": [25.999605462248432, 40.00039453775157], "tol": [0.005, 0.005], "function": 'T.INV.2T, SQRT',
     "working": "s_pred = 3√(1 + 1/15 + 16/160) ≈ 3.2404; 33 ± 7.000. The extra 1 is one observation's own scatter around the line, which no amount of data removes"},
    {"id": 'P2.8', "tier": 2, "group": 'Residual analysis',
     "q": "For P2.1's print runs, compute the standardized residual of every run. Which run has the largest in size, and is any beyond ±2?",
     "parts": ['the run with the largest (thousand pages)', 'its standardized residual'],
     "answer": [8, 1.7442605372282398], "tol": [0, 0.005], "function": 'STEYX, DEVSQ, SQRT',
     "working": 'h = 1/6 + (x − 6)²/40; standardized residual = e/(s√(1 − h)): -0.28, 0.00, 0.42, -1.25, 1.74, -0.84; the largest is the 8-thousand-page run, inside ±2'},
    {"id": 'P2.9', "tier": 2, "group": 'Outliers and influential observations',
     "q": 'Eight branches: staff = 4, 5, 5, 6, 6, 7, 8, 19. Compute the leverage of the branch with 19 staff, and compare it with the rule of thumb 6/n.',
     "parts": ['leverage h of the 19-staff branch', 'the rule of thumb 6/n'],
     "answer": [0.941358024691358, 0.75], "tol": [0.0005, 0.0005], "function": 'DEVSQ, AVERAGE',
     "working": 'x̄ = 7.5; Σ(x − x̄)² = 162; h = 1/8 + 11.5²/162 = 0.125 + 0.8164 ≈ 0.9414 > 0.75: high leverage'},
    {"id": 'P2.10', "tier": 2, "group": 'The coefficient of determination',
     "q": 'Summary statistics: r = 0.6, s_x = 5, s_y = 20, x̄ = 30, ȳ = 140. Find b1 and b0 without the raw data.',
     "parts": ['b₁', 'b₀'],
     "answer": [2.4, 68.0], "tol": [0.005, 0.005], "function": '—',
     "working": 'b1 = r s_y/s_x = 0.6 × 20/5 = 2.4; b0 = ȳ − b1 x̄ = 140 − 72 = 68'},
    {"id": 'P3.1', "tier": 3, "group": 'The model and least squares',
     "q": 'Server racks: ŷ = 2 + 0.06x (power in kW, load x in %), fitted to loads from 30 to 70%. What does the line give at 0% and at 100%, and how far should you trust each?',
     "parts": ['ŷ at 0% (kW)', 'ŷ at 100% (kW)'],
     "answer": [2.0000000000000004, 8.0], "tol": [0.005, 0.005], "function": 'FORECAST.LINEAR',
     "working": "ŷ(0) = 2.0 kW and ŷ(100) = 8.0 kW. Both are outside the data's 30–70%: extrapolations. Idle racks and racks at full load may behave differently (power saving, throttling); measure there before using them"},
    {"id": 'P3.2', "tier": 3, "group": 'Residual analysis',
     "q": "x = 1, 2, …, 8 and y = x² (1, 4, 9, …, 64). Fit a straight line. What is r², and what pattern do the residuals' signs show, from the smallest x to the largest?",
     "parts": ['r²', "the residuals' signs, smallest x first (8 signs, + or −, no spaces)"],
     "answer": [0.9529411764705882, '++−−−−++'], "tol": [0.0005, 0], "function": 'RSQ, SLOPE, INTERCEPT',
     "working": 'r² = 0.9529, high; residual signs + + − − − − + +: a U shape, so the relationship is curved. A high r² never proves the straight line is right; plot the residuals'},
    {"id": 'P3.3', "tier": 3, "group": 'Assumptions and testing for significance',
     "q": "Unit cost vs a supplier's list price, n = 30: b1 = 1.18, s_b1 = 0.07. Does cost rise one-for-one with list price? Test H0: β1 = 1 against Ha: β1 ≠ 1 at α = 0.05.",
     "parts": ['t', 'p-value (two-tailed)'],
     "answer": [2.5714285714285703, 0.015729032644963947], "tol": [0.0005, 0.0005], "function": 'T.DIST.2T',
     "working": 't = (b1 − 1)/s_b1 = 0.18/0.07 ≈ 2.571, df 28; p = T.DIST.2T(2.571, 28) ≈ 0.0157: reject; cost rises faster than list price'},
    {"id": 'P3.4', "tier": 3, "group": 'Assumptions and testing for significance',
     "q": 'A report gives only r² = 0.36 for a simple regression on n = 27 points. Work backwards: what are F, t and the p-value?',
     "parts": ['F', 't', 'p-value'],
     "answer": [14.062499999999998, 3.7499999999999996, 0.000938778615267255], "tol": [0.005, 0.0005, 5e-05], "function": 'F.DIST.RT',
     "working": 'F = (r²/1)/((1 − r²)/(n − 2)) = 0.36/(0.64/25) = 14.0625; t = √F = 3.7500; p = F.DIST.RT(F, 1, 25) ≈ 0.00094'},
    {"id": 'P3.5', "tier": 3, "group": 'Estimation and prediction',
     "q": "With s = 16, the half-width of the 95% prediction interval at x̄ is t × s × √(1 + 1/n). Compute it for n = 10 and for n = 1,000. Why doesn't it shrink towards zero?",
     "parts": ['half-width, n = 10', 'half-width, n = 1,000'],
     "answer": [38.69692065470539, 31.41319636750465], "tol": [0.005, 0.005], "function": 'T.INV.2T',
     "working": 'n = 10: 2.306 × 16 × √1.1 ≈ 38.70; n = 1,000: 1.962 × 16 × √1.001 ≈ 31.41. More data pins down the line, but one new value still scatters around it with standard deviation σ: about ±1.96σ at best'},
    {"id": 'P3.6', "tier": 3, "group": 'Estimation and prediction',
     "q": 'The warehouse (Try it tab): budget for a week of 58,000 orders so that the cost stays under budget 95% of the time. Use a one-sided 95% prediction bound.',
     "parts": ['the budget: one-sided 95% bound ($ thousands)'],
     "answer": [333.07922483159746], "tol": [0.005], "function": 'T.INV, STEYX, DEVSQ',
     "working": "ŷ = 301; s_pred = 16√(1 + 1/10 + 64/1024) ≈ 17.2511; one-sided t = T.INV(0.95, 8) ≈ 1.860; 301 + 1.860 × 17.251 ≈ 333.08 ($ thousands): lower than the two-sided interval's 340.78"},
    {"id": 'P3.7', "tier": 3, "group": 'Outliers and influential observations',
     "q": "P2.9's eight branches, monthly sales ($ thousands): 62, 71, 68, 78, 84, 91, 98, 150. Fit the line with and without the 19-staff branch. Is that branch influential?",
     "parts": ['b₁, all eight branches', 'b₁, without the 19-staff branch'],
     "answer": [5.685185185185185, 9.473684210526317], "tol": [0.0005, 0.0005], "function": 'SLOPE',
     "working": 'All eight: b1 ≈ 5.685; without it: b1 ≈ 9.474. One branch with leverage 0.94 moves the slope by about 3.8 ($ thousands per person): influential. Check whether it is the same kind of branch before using either line'},
    {"id": 'P3.8', "tier": 3, "group": 'Outliers and influential observations',
     "q": 'The warehouse plus an 11th week at exactly x̄ = 50 thousand orders, with cost keyed as 330. What happen to b1, b0 and s?',
     "parts": ['b₁', 'b₀', 's'],
     "answer": [4.5, 45.909090909090935, 25.579742616438352], "tol": [0.0005, 0.005, 0.005], "function": 'SLOPE, INTERCEPT, STEYX',
     "working": 'A point at x̄ has no pull on the slope: b1 stays 4.5. ȳ rises to 270.91, so b0 = 45.91. s jumps from 16 to 25.58, widening every interval; its standardized residual is 2.42'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": "The first idea: compare each engineer's salary with the engineers' average. How many engineers earn more than $20k below it, and how many of those have under five years' experience?",
     "parts": ['engineers more than $20k below the average', "of those, under five years' experience"],
     "answer": [23, 18], "tol": [0, 0], "function": 'AVERAGE, COUNTIFS',
     "working": 'average ≈ 102.93; 23 engineers below 82.93, 18 of them juniors: the average ignores experience'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": 'Fit salary on years for all 122 staff, directors included. Give b1, and the leverage of the two directors against the rule 6/n.',
     "parts": ['b₁, all 122 staff ($k a year)', 'leverage of the director with 31 years', 'leverage of the director with 34 years'],
     "answer": [4.018951895039968, 0.08957949945480023, 0.11609044239825919], "tol": [0.0005, 0.0005, 0.0005], "function": 'SLOPE, DEVSQ',
     "working": 'b1 ≈ 4.019; h ≈ 0.090 and 0.116, both above 6/122 ≈ 0.049; their standardized residuals ≈ 4.54 and 6.95: high leverage and far off the line'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": 'Fit the 120 engineers only: b1, b0, r² and s.',
     "parts": ['b₁ ($k a year)', 'b₀ ($k)', 'r²', 's ($k)'],
     "answer": [3.131424294432207, 69.01312251747515, 0.8096520877682994, 8.723810490170846], "tol": [0.0005, 0.005, 0.0005, 0.005], "function": 'SLOPE, INTERCEPT, RSQ, STEYX',
     "working": 'b1 ≈ 3.131 ($k a year of experience), b0 ≈ 69.01, r² ≈ 0.8097, s ≈ 8.724'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": "For the engineers' fit: t for the slope, and the 95% confidence interval for β1.",
     "parts": ['t', '95% interval for β₁: lower end', '95% interval for β₁: upper end'],
     "answer": [22.40351512862754, 2.8546341302387535, 3.4082144586256606], "tol": [0.005, 0.0005, 0.0005], "function": 'LINEST, T.INV.2T',
     "working": 's_b1 ≈ 0.1398; t ≈ 22.40; 3.131 ± 1.980 × 0.1398'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": "Standardized residuals for the engineers' fit: how many engineers fall below −2, and who are they?",
     "parts": ['how many below −2', 'their ids, in id order (a comma and a space between)'],
     "answer": [5, 'E003, E018, E041, E064, E089'], "tol": [0, 0], "function": 'STEYX, DEVSQ, COUNTIF',
     "working": 'standardized residual = e/(s√(1 − h)); below −2: E003 (3.5 years), E018 (7.2 years), E041 (10.4 years), E064 (13.8 years), E089 (16.5 years)'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": "What would it cost a year to raise each flagged engineer to the line (their predicted salary)? As a share of the engineers' payroll?",
     "parts": ['the cost a year ($ thousands)', "as a percent of the engineers' payroll"],
     "answer": [102.5208213211912, 0.8300071351640345], "tol": [0.05, 0.005], "function": 'SUM, TREND',
     "working": 'Σ(ŷ − y) over the flagged ≈ 102.52 ($k); payroll ≈ 12351.8: 0.83%'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": "A second question: a candidate with 6 years' experience asks for $112k. Give the 95% prediction interval for an engineer with 6 years. Is the ask in line with current pay?",
     "parts": ['95% prediction interval: lower end ($k)', '95% prediction interval: upper end ($k)'],
     "answer": [70.40284243591411, 105.20049413222269], "tol": [0.005, 0.005], "function": 'FORECAST.LINEAR, T.INV.2T',
     "working": 'ŷ ≈ 87.80; s_pred ≈ 8.786; 95% PI ≈ 70.40 to 105.20: $112k is above the interval. Paying it would put the new hire above almost every engineer with that experience, and open the kind of gap the review is fixing'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Brannock Engineering',
     "q": 'Write the recommendation. Anchor it: how many engineers would the confidence band for the mean have flagged instead, and why is that the wrong tool?',
     "parts": ['engineers below the confidence band'],
     "answer": [47], "tol": [0], "function": 'COUNTIFS, T.INV.2T',
     "working": "47 engineers sit below the confidence band: it describes the average salary at each experience, not one person's. Recommendation: review the 5 engineers below −2 (about $103k a year to bring them to the line); fit without the directors; check role, location and performance before any change"},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def split(text, q):
    """The typed answer, one part per comma; a last answer that is itself a list ("E003, E018") keeps its commas."""
    got = [read_part(p) for p in re.split(r"[,;]", text) if p.strip()]
    n, last = len(q["answer"]), q["answer"][-1]
    if len(got) > n and isinstance(last, str) and "," in last:
        got = got[:n - 1] + [", ".join(g if isinstance(g, str) else f"{g:g}" for g in got[n - 1:])]
    return got

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = split(box.value, q)
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't, signs and ids typed either way
assert right(read_part("80-3.5*12"), 38, 0.005) and not right(read_part("37"), 38, 0.005)
assert split("0.953, ++----++", Q["P3.2"])[1] in SAME["++−−−−++"] and right(split("0.953, ++−−−−++", Q["P3.2"])[1], "++−−−−++", None)

## Tier 1 · Warm-up

One idea, one step, small numbers: an intercept from the means, a prediction, a line through five points, a residual,
SSR and r², r from r², MSE and s, the t statistic for the slope, the two standard errors at x̄, and a standardized
residual.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

`stats.linregress(x, y)` fits the least-squares line: it returns the slope, the intercept, r, the slope's p-value and
its standard error. Note the order: **x first, then y**, the other way round from a spreadsheet's SLOPE(y, x). The
rest of the warm-ups are one line of arithmetic each. Each line below computes one warm-up answer and checks it
against the practice set.

In [ ]:
b0 = 80 - 3.5 * 12                                               # P1.1: the line passes through (x̄, ȳ)
yhat = 25 + 1.8 * 40                                             # P1.2
fit13 = stats.linregress([1, 2, 3, 4, 5], [3, 5, 4, 8, 10])      # P1.3: x first, then y
e = 112 - (20 + 3 * 30)                                          # P1.4: residual = y − ŷ
ssr = 500 - 125; r2 = ssr / 500                                  # P1.5
r = -math.sqrt(0.81)                                             # P1.6: r takes the slope's sign
mse = 72 / (10 - 2); s = math.sqrt(mse)                          # P1.7
t, df = 2.4 / 0.8, 14 - 2                                        # P1.8
s_yhat, s_pred = 4 * math.sqrt(1 / 25), 4 * math.sqrt(1 + 1 / 25)   # P1.9: at x̄ the distance term is 0
std = 6 / (4 * math.sqrt(1 - 0.1))                               # P1.10

print(f"b0 {b0}   ŷ {yhat}   line through five points: b1 {fit13.slope:.4f}, b0 {fit13.intercept:.4f}   residual {e}")
print(f"SSR {ssr}, r² {r2}   r {r}   MSE {mse}, s {s}   t {t} (df {df})   s_ŷ {s_yhat:.3f}, s_pred {s_pred:.3f}   "
      f"standardized {std:.3f}")
for k, v in (("P1.1", [b0]), ("P1.2", [yhat]), ("P1.3", [fit13.slope, fit13.intercept]), ("P1.4", [e]),
             ("P1.5", [ssr, r2]), ("P1.6", [r]), ("P1.7", [mse, s]), ("P1.8", [t, df]), ("P1.9", [s_yhat, s_pred]),
             ("P1.10", [std])):
    assert agrees(k, v), k

### The warehouse, in three lines

The course's warehouse: ten weeks of orders (thousands) and operating cost ($ thousands). `linregress` gives the line
the video worked out by hand: a slope of 4.5 (each extra thousand orders adds about $4,500) and an intercept of 40.

In [ ]:
orders = [41, 51, 56, 37, 65, 44, 59, 35, 49, 63]
cost = [247, 253, 295, 190, 355, 235, 294, 212, 250, 319]
fit = stats.linregress(orders, cost)
fit

In [ ]:
print("slope", fit.slope, " intercept", fit.intercept)
print("r² =", fit.rvalue ** 2, "  the slope's standard error", fit.stderr, "  its p-value", fit.pvalue)
assert round(fit.slope, 9) == 4.5 and round(fit.intercept, 9) == 40                       # the video's numbers
assert round(fit.rvalue ** 2, 4) == 0.9101 and round(fit.stderr, 9) == 0.5 and round(fit.pvalue * 1e5, 2) == 1.85
assert round(fit.intercept_stderr, 2) == 25.51

### Slide the week you're budgeting for

The line predicts next week's cost at x* thousand orders: ŷ = 40 + 4.5x*. Two intervals sit around it. The narrow
**confidence interval** (SKY) is for the *average* cost of all weeks with x* orders; the wide **prediction interval**
(VIOLET) is for *one* week's cost. At x* = 58: ŷ = 301, the confidence interval about 286 to 316, the prediction
interval about 261 to 341. Slide x* away from x̄ = 50 and watch both widen; the prediction band never gets much
narrower than about ±2s.

In [ ]:
def intervals(x, y, xs, conf=0.95):
    """ŷ at xs, the confidence interval for the mean of y there, and the prediction interval for one y."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x); f = stats.linregress(x, y)
    s = math.sqrt(((y - (f.intercept + f.slope * x)) ** 2).sum() / (n - 2))
    sxx = ((x - x.mean()) ** 2).sum()
    t = stats.t.ppf(1 - (1 - conf) / 2, n - 2)
    yh = f.intercept + f.slope * xs
    s_yhat = s * math.sqrt(1 / n + (xs - x.mean()) ** 2 / sxx)
    s_pred = s * math.sqrt(1 + 1 / n + (xs - x.mean()) ** 2 / sxx)
    return yh, (yh - t * s_yhat, yh + t * s_yhat), (yh - t * s_pred, yh + t * s_pred)

def budget_picture(x_star=58.0):
    xs = np.linspace(25, 80, 200)
    band = [intervals(orders, cost, v) for v in xs]
    ci = np.array([b[1] for b in band], float); pi = np.array([b[2] for b in band], float)
    yh, (c0, c1), (p0, p1) = intervals(orders, cost, x_star)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.fill_between(xs, pi[:, 0], pi[:, 1], color=VIOLET, alpha=0.18, lw=0)
    ax.fill_between(xs, ci[:, 0], ci[:, 1], color=SKY, alpha=0.35, lw=0)
    ax.plot(xs, fit.intercept + fit.slope * xs, color=SKY)
    ax.scatter(orders, cost, s=55, color=CHALK, zorder=3)
    ax.plot([x_star] * 2, [p0, p1], color=VIOLET, lw=4); ax.plot([x_star] * 2, [c0, c1], color=SKY, lw=7)
    ax.scatter([x_star], [yh], s=90, color=GOLD, zorder=4)
    ax.set_xlim(25, 80); ax.set_ylim(120, 460); ax.set_xlabel("orders (thousands)"); ax.set_ylabel("cost ($ thousands)")
    fig.suptitle(f"x* = {x_star:g}: ŷ = {yh:.1f}; mean {c0:.1f} to {c1:.1f}; one week {p0:.1f} to {p1:.1f}", x=0.08,
                 ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

_yh, _ci, _pi = intervals(orders, cost, 58)                                               # the video's numbers
assert round(_yh, 9) == 301 and [round(v, 1) for v in _ci] == [286.1, 315.9] and [round(v, 1) for v in _pi] == [261.2, 340.8]

In [ ]:
interact(budget_picture, x_star=FloatSlider(value=58, min=25, max=80, step=0.5, description="x* (orders, k)"));

## Tier 2 · Core

The module's standard problems, several steps each: a least-squares line and what its slope means, the sums of
squares, the t test for the slope, an interval for β₁, the ANOVA table, a confidence and a prediction interval,
standardized residuals, leverage, and b₁ and b₀ from summary statistics.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**The fit and its test.** `linregress` returns the slope's standard error (`stderr`) and its two-tailed p-value
(`pvalue`) for H₀: β₁ = 0. **The sums of squares** are three lines of numpy: SST from y's deviations, SSE from the
residuals, SSR from the fitted values. **The t and F distributions**: `stats.t.ppf(0.975, df)` is a spreadsheet's
T.INV.2T(0.05, df), `stats.t.sf` a right tail, and `stats.f.sf(F, 1, df)` is F.DIST.RT(F, 1, df). **Residuals and
leverage**: h = 1/n + (x − x̄)²/Σ(x − x̄)², and a standardized residual is e/(s√(1 − h)).

In [ ]:
pages = np.array([2, 4, 5, 7, 8, 10]); minutes = np.array([26, 37, 43, 51, 61, 68])        # P2.1–P2.3, P2.8
f = stats.linregress(pages, minutes)
yhat = f.intercept + f.slope * pages; e = minutes - yhat
sst, sse = ((minutes - minutes.mean()) ** 2).sum(), (e ** 2).sum()
ssr = ((yhat - minutes.mean()) ** 2).sum()
n = len(pages); s = math.sqrt(sse / (n - 2)); sxx = ((pages - pages.mean()) ** 2).sum()
print(f"print runs: ŷ = {f.intercept:.3f} + {f.slope:.4f}x;  SST {sst:.2f} = SSR {ssr:.2f} + SSE {sse:.2f};  "
      f"r² {f.rvalue ** 2:.4f};  t {f.slope / f.stderr:.2f}, p {f.pvalue:.6f}")
assert agrees("P2.1", [f.slope, f.intercept]) and agrees("P2.2", [sst, sse, ssr, f.rvalue ** 2, f.rvalue])
assert agrees("P2.3", [s, f.stderr, f.slope / f.stderr, f.pvalue]) and abs(f.stderr - s / math.sqrt(sxx)) < 1e-12

h = 1 / n + (pages - pages.mean()) ** 2 / sxx                                             # P2.8: leverage, then
std = e / (s * np.sqrt(1 - h))                                                             # standardized residuals
k = int(np.argmax(np.abs(std)))
print("standardized residuals:", np.round(std, 2), " largest in size: the", pages[k], "thousand-page run")
assert agrees("P2.8", [pages[k], std[k]])

q = stats.t.ppf(0.975, 18 - 2)                                                             # P2.4: interval for β₁
assert agrees("P2.4", [0.85 - q * 0.21, 0.85 + q * 0.21])
msr, mse = 840 / 1, 360 / (20 - 2); F = msr / mse                                          # P2.5: the ANOVA table
assert agrees("P2.5", [msr, mse, F, stats.f.sf(F, 1, 18)])
q = stats.t.ppf(0.975, 15 - 2); yh = 5 + 2 * 14                                             # P2.6, P2.7: at x* = 14
s_yhat, s_pred = 3 * math.sqrt(1 / 15 + (14 - 10) ** 2 / 160), 3 * math.sqrt(1 + 1 / 15 + (14 - 10) ** 2 / 160)
assert agrees("P2.6", [yh - q * s_yhat, yh + q * s_yhat]) and agrees("P2.7", [yh - q * s_pred, yh + q * s_pred])
staff = np.array([4, 5, 5, 6, 6, 7, 8, 19])                                                # P2.9: leverage
h_big = 1 / len(staff) + (19 - staff.mean()) ** 2 / ((staff - staff.mean()) ** 2).sum()
assert agrees("P2.9", [h_big, 6 / len(staff)])
b1 = 0.6 * 20 / 5                                                                          # P2.10: from r, s_x, s_y
assert agrees("P2.10", [b1, 140 - b1 * 30])

The ANOVA table for the warehouse, worked the same way: SSR 20,736 and SSE 2,048 add to SST 22,784; MSE = 2,048/8 =
256, so s = 16; F = 20,736/256 = 81, which is t² = 9²; and both tests give the same p-value, about 0.00002.

In [ ]:
w_yhat = fit.intercept + fit.slope * np.array(orders)
w_sse = ((np.array(cost) - w_yhat) ** 2).sum(); w_ssr = ((w_yhat - np.mean(cost)) ** 2).sum()
w_F = w_ssr / (w_sse / 8)
anova = pd.DataFrame({"sum of squares": [w_ssr, w_sse, w_ssr + w_sse], "df": [1, 8, 9],
                      "mean square": [w_ssr / 1, w_sse / 8, np.nan], "F": [w_F, np.nan, np.nan],
                      "p-value": [stats.f.sf(w_F, 1, 8), np.nan, np.nan]}, index=["Regression", "Error", "Total"])
print(anova.round(6).to_string())
assert round(w_ssr, 6) == 20736 and round(w_sse, 6) == 2048 and round(w_F, 6) == 81
assert abs(stats.f.sf(w_F, 1, 8) - fit.pvalue) < 1e-12 and round((fit.slope / fit.stderr) ** 2, 6) == 81

### One more week: which points can tilt the line?

Add an eleventh week to the warehouse. Put it at x̄ = 50 thousand orders and drag its cost: the line moves up and
down but its slope stays 4.5, because a point at x̄ has no leverage on the slope. Now put it far to the right, at 95
thousand orders (a holiday peak). On the line, at 468, the slope barely moves (4.51); with overtime, at 525, it jumps
to 5.32, and the point's own residual stays small (standardized about 1.76): it has dragged the line to itself.

In [ ]:
def week_picture(x11=95.0, y11=468.0):
    x = np.array(orders + [x11], float); y = np.array(cost + [y11], float)
    f11 = stats.linregress(x, y)
    n = len(x); e11 = y - (f11.intercept + f11.slope * x)
    s11 = math.sqrt((e11 ** 2).sum() / (n - 2)); sxx11 = ((x - x.mean()) ** 2).sum()
    h11 = 1 / n + (x - x.mean()) ** 2 / sxx11
    std11 = e11[-1] / (s11 * math.sqrt(1 - h11[-1]))
    xs = np.linspace(25, 105, 50)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(xs, fit.intercept + fit.slope * xs, color=DIM, ls="--", lw=1.6, label="ten weeks: slope 4.50")
    ax.plot(xs, f11.intercept + f11.slope * xs, color=SKY, label=f"with week 11: slope {f11.slope:.2f}")
    ax.scatter(orders, cost, s=55, color=CHALK, zorder=3)
    ax.scatter([x11], [y11], s=110, color=CORAL if h11[-1] > 6 / n else GOLD, zorder=4)
    ax.set_xlim(25, 105); ax.set_ylim(120, 620); ax.set_xlabel("orders (thousands)"); ax.set_ylabel("cost ($ thousands)")
    ax.legend(loc="upper left", frameon=False)
    fig.suptitle(f"week 11 at ({x11:g}, {y11:g}): leverage {h11[-1]:.3f} (rule 6/n = {6 / n:.3f}), standardized "
                 f"residual {std11:.2f}", x=0.08, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

def _slope11(x11, y11): return stats.linregress(orders + [x11], cost + [y11]).slope
assert round(_slope11(50, 330), 9) == 4.5 and round(_slope11(95, 468), 2) == 4.51 and round(_slope11(95, 525), 2) == 5.32

In [ ]:
interact(week_picture, x11=FloatSlider(value=95, min=30, max=100, step=1, description="orders"),
         y11=FloatSlider(value=468, min=150, max=600, step=1, description="cost"));

## Tier 3 · Stretch

Combine ideas, choose the method, spot the trap, work backwards: trusting the line outside the data, a high r² on a
curve, testing β₁ = 1 instead of 0, F, t and p from r² alone, why a prediction interval can't shrink to nothing, a
one-sided budget, one branch that moves the slope, and an outlier at x̄.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

A high r² never proves a straight line is right: fit y = x² and the residuals' signs run + then − then +, a U.
Testing β₁ = 1 is the same t test with 1 subtracted. From r² alone, F = r²(n − 2)/(1 − r²) and t = √F. A one-sided
bound uses `stats.t.ppf(0.95, df)` (a spreadsheet's T.INV(0.95, df)). And refitting without one point is one more
call to `linregress`.

In [ ]:
racks = stats.linregress([30, 40, 50, 60, 70], [3.9, 4.2, 5.0, 5.8, 6.1])                 # P3.1: outside 30–70%
assert agrees("P3.1", [racks.intercept, racks.intercept + racks.slope * 100])

x = np.arange(1, 9); y = x ** 2                                                            # P3.2: a curve
fq = stats.linregress(x, y)
signs = "".join("+" if v > 0 else "−" for v in y - (fq.intercept + fq.slope * x))
print(f"y = x²: r² = {fq.rvalue ** 2:.4f}, residual signs {signs}")
assert agrees("P3.2", [fq.rvalue ** 2, signs])

t = (1.18 - 1) / 0.07                                                                      # P3.3: H₀ β₁ = 1
assert agrees("P3.3", [t, 2 * stats.t.sf(abs(t), 28)])
F = 0.36 * (27 - 2) / (1 - 0.36)                                                           # P3.4: from r² alone
assert agrees("P3.4", [F, math.sqrt(F), stats.f.sf(F, 1, 25)])
half = lambda n: stats.t.ppf(0.975, n - 2) * 16 * math.sqrt(1 + 1 / n)                    # P3.5
print(f"prediction half-width at x̄, s = 16: n = 10 → {half(10):.2f}; n = 1,000 → {half(1000):.2f}; "
      f"n = 1,000,000 → {half(10 ** 6):.2f} (never below 1.96 × 16 = 31.36)")
assert agrees("P3.5", [half(10), half(1000)])

xw, yw = np.array(orders, float), np.array(cost, float)                                    # P3.6: a one-sided budget
s_w = math.sqrt(((yw - (fit.intercept + fit.slope * xw)) ** 2).sum() / 8)
sp58 = s_w * math.sqrt(1 + 1 / 10 + (58 - xw.mean()) ** 2 / ((xw - xw.mean()) ** 2).sum())
budget = fit.intercept + fit.slope * 58 + stats.t.ppf(0.95, 8) * sp58
print(f"P3.6: budget {budget:.2f} ($ thousands) covers 95% of weeks with 58,000 orders")
assert agrees("P3.6", [budget])

sales = [62, 71, 68, 78, 84, 91, 98, 150]                                                  # P3.7: with and without
assert agrees("P3.7", [stats.linregress(staff, sales).slope, stats.linregress(staff[:7], sales[:7]).slope])
f11 = stats.linregress(orders + [50], cost + [330])                                        # P3.8: an outlier at x̄
e11 = np.array(cost + [330]) - (f11.intercept + f11.slope * np.array(orders + [50]))
assert agrees("P3.8", [f11.slope, f11.intercept, math.sqrt((e11 ** 2).sum() / 9)])

## Tier 4 · The case: Brannock Engineering's pay review

Brannock Engineering is an engineering consultancy (a fictional company; our own data). Before its annual pay round
it asks: **are any engineers paid well below what their experience predicts, and what would it cost to fix?** Here
are its 122 staff, one row each:

| column | what it is |
|---|---|
| `id` | employee id, E001–E122 |
| `role` | engineer, or director (pay set by the board) |
| `years` | years of professional experience |
| `salary` | annual salary, $ thousands |

The next cell builds the table (the same 122 rows as the spreadsheet's Case study tab) and checks a few of its
numbers.

In [ ]:
# @CASE
# years of experience and salary ($ thousands), in id order: 120 engineers, then the 2 directors
YEARS = [5.1, 11.1, 3.5, 12.3, 12.7, 1.8, 0.8, 16.8, 5.6, 5.1, 19.9, 9.7, 16.8, 9.8, 13, 3.4, 12.9, 7.2, 10.7, 15, 13.6,
         1.7, 15.3, 12, 6.4, 1.1, 17.4, 9.7, 14.5, 17.6, 14.4, 18.5, 8.2, 16.1, 9.2, 18.7, 17.6, 2.4, 3.2, 4.7, 10.4, 9,
         12.7, 6.4, 10.4, 8, 7.3, 11.9, 11.9, 18.1, 13.8, 18.6, 17.2, 19.8, 13.6, 3.7, 17.3, 19.3, 18.1, 11.6, 14.4, 4.6,
         16.7, 13.8, 6.1, 1.7, 17.2, 19.8, 2.2, 16.1, 8.5, 3.4, 6.2, 15.5, 17.5, 1.4, 12.5, 1.4, 14.5, 7, 17.7, 19.6,
         10.4, 20, 6.5, 2, 12.2, 1.1, 16.5, 8.5, 12.4, 3.5, 1.3, 17.4, 6.6, 19.2, 18, 7.9, 9.5, 10.6, 13.1, 12.1, 11.4,
         12.6, 18.8, 10.4, 8.9, 14.5, 5.1, 6.4, 19.6, 10.7, 11.2, 0.7, 8.6, 11.8, 0.9, 12.5, 12.8, 1.7, 31, 34]
SALARY = [80.1, 99.1, 60.7, 102.6, 107.1, 63.1, 75.4, 124.2, 78.7, 67.9, 133.6, 109.9, 117.9, 97.5, 107.1, 86.1, 104, 71,
          101.8, 125.4, 113.8, 73.6, 107.1, 102.9, 88.4, 78.8, 127.6, 95.5, 119.7, 134.2, 114.9, 125.7, 93.1, 128, 103.7,
          122.4, 129.3, 73.8, 74.2, 95, 79.3, 93, 111.3, 94.5, 98.2, 94.6, 98.7, 93.8, 110.7, 133.8, 118.2, 118.8, 127.6,
          126.5, 118.1, 86.7, 127.1, 125.6, 123.2, 114, 108.9, 88.7, 127.5, 92.7, 108.7, 76, 142.2, 117.2, 59.1, 129.4,
          102.3, 78.4, 89.4, 104.4, 121, 66.2, 108.2, 81.6, 116.8, 95.4, 121.1, 129.2, 104.2, 131.7, 100.9, 69.4, 124.2,
          65.7, 99.8, 91, 122.9, 82.3, 77.3, 131.6, 86, 123.1, 111.4, 105, 94.8, 99.2, 111.7, 124.6, 92.7, 112.3, 127,
          107.5, 84.1, 113.2, 93, 103, 145.5, 97.4, 106.3, 71.4, 86.5, 96.2, 78.9, 112, 109.8, 85.2, 248, 292]
# @END CASE
df = pd.DataFrame({"id": [f"E{k + 1:03d}" for k in range(len(YEARS))],
                   "role": ["engineer"] * 120 + ["director"] * 2, "years": YEARS, "salary": SALARY})

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'n_engineers': 120,
 'avg': 102.93166666666667,
 'below_avg': 23,
 'below_avg_juniors': 18,
 'all_b1': 4.018951895039968,
 'all_s': 14.475347784422352,
 'h_directors': [0.08957949945480023, 0.11609044239825919],
 'std_directors': [4.5397117412873635, 6.954459227934176],
 'eng_b1': 3.131424294432207,
 'eng_b0': 69.01312251747515,
 'eng_r2': 0.8096520877682994,
 'eng_s': 8.723810490170846,
 'eng_t': 22.40351512862754,
 'flag_e': ['E003', 'E018', 'E041', 'E064', 'E089'],
 'gap_total': 102.5208213211912,
 'payroll': 12351.800000000001,
 'ci_flag': 47,
 'above_e': 3,
 'offer_pi': (70.40284243591411, 105.20049413222269)}
# @END STATS
S = CASE_STATS
assert len(df) == 122 and (df["role"] == "engineer").sum() == S["n_engineers"] == 120
assert abs(df.loc[df["role"] == "engineer", "salary"].mean() - S["avg"]) < 1e-9
print(f"{len(df)} staff, {df.shape[1]} columns: the same data as the course's case.")
df.head(8)

### How you'd do this in Python

With the staff in a DataFrame, each step of the case is a line or two: `linregress` on the engineers only (a boolean
mask, `df["role"] == "engineer"`), the residuals and leverages with numpy, and the flag where the standardized
residual is below −2. The cells below reproduce the video's case: the two tempting shortcuts that mislead, then the
right one.

In [ ]:
def fit_table(d):
    """The least-squares fit of salary on years for the rows of d, with each row's ŷ, residual, leverage and
    standardized residual."""
    x, y = d["years"].to_numpy(float), d["salary"].to_numpy(float)
    f = stats.linregress(x, y)
    n = len(x); yh = f.intercept + f.slope * x; e = y - yh
    s = math.sqrt((e ** 2).sum() / (n - 2)); sxx = ((x - x.mean()) ** 2).sum()
    h = 1 / n + (x - x.mean()) ** 2 / sxx
    out = d.assign(yhat=yh, resid=e, leverage=h, std=e / (s * np.sqrt(1 - h)))
    return f, s, sxx, out

eng = df[df["role"] == "engineer"]
avg = eng["salary"].mean()                                                                 # shortcut 1: the average
low = eng[eng["salary"] < avg - 20]
print(f"1 · the average, ${avg:.1f}k: {len(low)} engineers more than $20k below it, "
      f"{(low['years'] < 5).sum()} of them juniors (the average ignores experience)")
fa, sa, _, all_ = fit_table(df)                                                            # shortcut 2: everyone
print(f"2 · all 122 staff: slope {fa.slope:.2f}, s {sa:.1f}; the directors' leverage "
      f"{', '.join(f'{v:.2f}' for v in all_['leverage'].tail(2))} (rule 6/n = {6 / 122:.3f}), standardized "
      f"{', '.join(f'{v:.1f}' for v in all_['std'].tail(2))}; engineers below −2: {(all_['std'].head(120) < -2).sum()}")
fe, se_, sxe, en = fit_table(eng)                                                          # the right one
flag = en[en["std"] < -2]
print(f"3 · engineers only: ŷ = {fe.intercept:.1f} + {fe.slope:.2f}x, r² {fe.rvalue ** 2:.2f}, s {se_:.1f}; "
      f"below −2: {', '.join(flag['id'])} ({', '.join(f'{v:g}' for v in flag['years'])} years)")
print(f"   to bring them to the line: ${(flag['yhat'] - flag['salary']).sum():.1f}k a year")
assert len(low) == 23 and (low["years"] < 5).sum() == 18 and (all_["std"].head(120) < -2).sum() == 0
assert round(fa.slope, 2) == 4.02 and round(sa, 1) == 14.5 and round(fe.slope, 2) == 3.13 and round(fe.intercept, 1) == 69.0
assert list(flag["id"]) == S["flag_e"] and round((flag["yhat"] - flag["salary"]).sum(), 1) == 102.5

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
ax.axhline(-2, color=CORAL, ls="--", lw=1.6); ax.axhline(2, color=DIM, ls="--", lw=1.2); ax.axhline(0, color=DIM, lw=1)
ax.scatter(en["years"], en["std"], s=22, color=CHALK, alpha=0.8)
ax.scatter(flag["years"], flag["std"], s=90, color=GOLD, zorder=4)
for _, row in flag.iterrows():
    ax.annotate(row["id"], (row["years"], row["std"]), textcoords="offset points", xytext=(6, -14), color=GOLD, fontsize=11)
ax.set_xlabel("years of experience"); ax.set_ylabel("standardized residual"); ax.set_ylim(-3.6, 3.2)
fig.suptitle(f"The engineers' line: {len(flag)} engineers below −2, about ${(flag['yhat'] - flag['salary']).sum():.0f}k "
             "a year to bring to the line", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the slope's interval, the cost of the fix as a share of payroll,
a candidate with six years who asks for $112k, and how many engineers the wrong band would have flagged, ending in
your recommendation. Use the DataFrame `df` in a cell of your own (press **+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `df` with pandas, numpy and scipy and checks it
against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
mine = {}
mine["P4.1"] = [len(low), (low["years"] < 5).sum()]
mine["P4.2"] = [fa.slope, *all_["leverage"].tail(2)]
mine["P4.3"] = [fe.slope, fe.intercept, fe.rvalue ** 2, se_]
q = stats.t.ppf(0.975, len(eng) - 2)
mine["P4.4"] = [fe.slope / fe.stderr, fe.slope - q * fe.stderr, fe.slope + q * fe.stderr]
mine["P4.5"] = [len(flag), ", ".join(flag["id"])]
gap = (flag["yhat"] - flag["salary"]).sum()
mine["P4.6"] = [gap, gap / eng["salary"].sum() * 100]
xb = eng["years"].mean()
yh6 = fe.intercept + fe.slope * 6
sp6 = se_ * math.sqrt(1 + 1 / len(eng) + (6 - xb) ** 2 / sxe)
mine["P4.7"] = [yh6 - q * sp6, yh6 + q * sp6]
ci_low = en["yhat"] - q * se_ * np.sqrt(1 / len(eng) + (en["years"] - xb) ** 2 / sxe)
mine["P4.8"] = [(en["salary"] < ci_low).sum()]

for k, got in mine.items():
    got = [v if isinstance(v, str) else float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {v if isinstance(v, str) else show(round(v, 6))}" for p, v in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")
print(f"The candidate: the 95% prediction interval at 6 years runs to ${mine['P4.7'][1]:.1f}k, so the $112k ask is above it.")

## Sources
Every exercise, data set and company here is our own (Brannock Engineering is fictional). The chapter order follows a
standard first course in business statistics (simple linear regression); the definitions agree with OpenStax,
Introductory Business Statistics 2e, chapter 13, "Linear Regression and Correlation" (retrieved 2026-11-09). The
leverage rule 6/n (three times the average leverage 2/n) follows Penn State STAT 501, lesson 11.2 (retrieved
2026-11-09).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs06_simple_regression/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs06_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert all((a == b) if isinstance(b, str) else abs(a - float(b)) < 1e-9
                   for a, b in zip(_q["answer"], _flat(_e["answer"]))) and len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    assert _P.CASE["years"] == df["years"].tolist() and _P.CASE["salary"] == df["salary"].tolist()
    assert _P.CASE["role"] == df["role"].tolist() and _P.CASE["id"] == df["id"].tolist()
    print("matches the course's practice set")